# PhishScamSense — Neural Pipeline Training (Google Colab)
Run each cell in order. GPU runtime required.

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## Step 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — change runtime to GPU!')
assert torch.cuda.is_available(), 'Please enable GPU: Runtime → Change runtime type → T4 GPU'

## Step 2 — Clone project from GitHub

In [ ]:
!git clone https://github.com/penanamtomat/PhishScamSense.git
%cd PhishScamSense
!ls

## Step 3 — Install dependencies

In [ ]:
# PyTorch is already installed on Colab with CUDA — just install the rest
!pip install -q transformers==4.47.0 xgboost==2.1.3 scikit-learn==1.6.0 tldextract beautifulsoup4

## Step 4 — Move manually uploaded data to correct directories
Make sure you have already uploaded these files via the Colab file panel (📁 left sidebar):
- `benign_domains.csv`
- `phishing_domains.csv`
- `malware_domains.csv`
- `spam_domains.csv`
- `false_positives_benign.csv`

In [ ]:
import os, shutil

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/feedback', exist_ok=True)

raw_files = ['benign_domains.csv', 'phishing_domains.csv', 'malware_domains.csv', 'spam_domains.csv']

for fname in raw_files:
    src = f'/content/{fname}'
    dst = f'data/raw/{fname}'
    if os.path.exists(src):
        shutil.move(src, dst)
        print(f'✅ Moved {fname} → data/raw/')
    elif os.path.exists(dst):
        print(f'✅ {fname} already in data/raw/')
    else:
        print(f'❌ {fname} not found — please upload it first')

fp_src = '/content/false_positives_benign.csv'
fp_dst = 'data/feedback/false_positives_benign.csv'
if os.path.exists(fp_src):
    shutil.move(fp_src, fp_dst)
    print('✅ Moved false_positives_benign.csv → data/feedback/')
elif os.path.exists(fp_dst):
    print('✅ false_positives_benign.csv already in data/feedback/')
else:
    print('❌ false_positives_benign.csv not found — please upload it first')

print('\ndata/raw contents:')
!ls -lh data/raw/
print('\ndata/feedback contents:')
!ls -lh data/feedback/

## Step 5 — Run neural training

In [ ]:
!python -m ml.src.training.train \
    --mode neural \
    --data-dir data/raw \
    --output-dir ml/exports \
    --feedback-dir data/feedback \
    --max-benign 100000 \
    --epochs 5 \
    --batch-size 64 \
    --lr 1e-4

## Step 6 — Verify output

In [ ]:
!ls -lh ml/exports/
import json
with open('ml/exports/model_info.json') as f:
    print(json.dumps(json.load(f), indent=2))

## Step 7 — Download trained models

In [ ]:
import shutil
from google.colab import files

# Zip the exports folder
shutil.make_archive('phishscamsense_models', 'zip', 'ml/exports')
print('Downloading phishscamsense_models.zip...')
files.download('phishscamsense_models.zip')

## After downloading
Extract the zip and upload to your VPS:
```bash
# Extract locally
# Then push to VPS:
scp fusion_model_latest.pt root@72.62.243.228:/root/project-url-phishing/PhishScamSense/ml/exports/fusion_model.pt
scp xgb_classifier_latest.pkl root@72.62.243.228:/root/project-url-phishing/PhishScamSense/ml/exports/xgb_classifier.pkl
scp model_info.json root@72.62.243.228:/root/project-url-phishing/PhishScamSense/ml/exports/model_info.json

# Restart backend
ssh root@72.62.243.228 'cd /root/project-url-phishing/PhishScamSense && docker compose restart backend'
```